In [3]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from google.colab import drive

drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/ML Project/final_mta_ml_matrix.csv'

Mounted at /content/drive


In [4]:
df = pd.read_csv(file_path)
df['transit_timestamp'] = pd.to_datetime(df['transit_timestamp'])

# 1. Chronological Split (Train on 2022-2023, Test on 2024)
train_df = df[df['transit_timestamp'] < '2024-01-01']
test_df = df[df['transit_timestamp'] >= '2024-01-01']

# 2. Separate Features (X) and Target (y)
X_train = train_df.drop(columns=['transit_timestamp', 'ridership'])
y_train = train_df['ridership']

X_test = test_df.drop(columns=['transit_timestamp', 'ridership'])
y_test = test_df['ridership']

print(f"Training rows: {len(X_train)} | Testing rows: {len(X_test)}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ML Project/final_mta_ml_matrix.csv'

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# 1. Initialize the model
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)

# 2. Train the model
print("Training Random Forest...")
rf_model.fit(X_train, y_train)

# 3. Predict and Evaluate
rf_predictions = rf_model.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_predictions)

print(f"Random Forest MAE: {rf_mae:.2f} riders")

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# 1. Initialize the Gradient Booster
gb_model = GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42)

# 2. Train the model
print("Training Gradient Booster...")
gb_model.fit(X_train, y_train)

# 3. Predict and Evaluate
gb_predictions = gb_model.predict(X_test)
gb_mae = mean_absolute_error(y_test, gb_predictions)

print(f"Gradient Boosting MAE: {gb_mae:.2f} riders")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract importances from the trained Random Forest
importances = rf_model.feature_importances_
feature_names = X_train.columns

# Sort them descending
indices = np.argsort(importances)[::-1]

# Plot the top 10 most important features
plt.figure(figsize=(10, 6))
plt.title("Top 10 Drivers of Subway Ridership (Random Forest)")
plt.bar(range(10), importances[indices][:10], align="center")
plt.xticks(range(10), [feature_names[i] for i in indices[:10]], rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor
import numpy as np

# 1. Define the grid of parameters to test
param_dist = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'subsample': [0.8, 0.9, 1.0]
}

# 2. Set up the Search
gb_base = GradientBoostingRegressor(random_state=42)
random_search = RandomizedSearchCV(
    estimator=gb_base,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)

# 3. Run the search (This will take a few minutes)
print("Searching for optimal Gradient Boosting parameters...")
random_search.fit(X_train, y_train)

print(f"Best Parameters: {random_search.best_params_}")

# 4. Evaluate the optimized model
optimized_gb = random_search.best_estimator_
opt_predictions = optimized_gb.predict(X_test)
print(f"Optimized GBM MAE: {mean_absolute_error(y_test, opt_predictions):.2f}")

In [ ]:
!pip install shap

In [ ]:
import shap

# 1. Initialize the SHAP explainer on your best model (e.g., the Random Forest)
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test.sample(2000, random_state=42))

# 2. Generate the Summary Plot
shap.summary_plot(shap_values, X_test.sample(2000, random_state=42))